# 27B donor -> {9B, 2B} recipients: DIFFICULTY-MATCHED conferral

## The confound this notebook fixes

Two completed runs share the donor `google/gemma-2-27b`:

| | 27B -> 9B | 27B -> 2B |
|---|---|---|
| unsolvable bin n | 240 | 729 |
| first-token conferral | 0.725 | 0.778 |
| full-answer conferral | 0.120 | 0.054 |

Read naively this is a dissociation: the **weaker** recipient gets **more** first-token
conferral and **less** full-answer conferral, which would suggest the first token is
transferred content (capability-independent) while later digits are the recipient's own
computation (capability-dependent).

**But the two bins are not the same problems.** The "unsolvable bin" is defined per
recipient as *the eval problems that recipient fails natively at the first token*. The 9B
fails only the hardest ~13% of donor-solved problems; the 2B fails the hardest ~41%. The
9B is therefore being scored on a strictly harder subset. Any difference between the two
columns is a mixture of the recipient effect and a difficulty effect, and nothing in the
saved aggregates can separate them.

**The fix.** Evaluate both recipients against the *same* donor states and export
**per-problem** outcomes, then compare both recipients on the **intersection bin** -- the
problems that BOTH recipients fail natively. On that identical problem set, difficulty is
held fixed by construction and the remaining difference is the recipient effect. Because
the comparison is now paired on the same items, the notebook also reports an exact
**McNemar** test alongside Wilson intervals.

The previous runs saved only aggregates, so the intersection cannot be recovered from
them -- this notebook must recompute both recipients, which is why the donor is loaded
once and shared.

## Run order

1. **CELL 1** -- install. **RESTART THE KERNEL**, then continue at CELL 2.
2. CELL 2-5b -- config, HF login (paste your token), helpers.
3. **CELL 6** -- load the 27B donor in bf16. Slow: ~54 GB of weights to download once.
4. **CELL 7** -- generate the arithmetic problems, run the donor ONCE, fix the
   donor-solved set, emit `donor_solved_exprs_sha256`.
5. CELL 8 -- the recipient engine (functions only; nothing runs).
6. **CELL 9** -- Phase A: load `gemma-2-9b`, confirm layers, train the map, evaluate.
7. **CELL 10** -- free recipient A completely (asserted, with a printed VRAM check).
8. **CELL 11** -- Phase B: load `gemma-2-2b`, confirm layers, train the map, evaluate.
9. **CELL 12** -- the difficulty-matched analysis (the point of the notebook).
10. **CELL 13** -- save `gemma27b_difficulty_matched_results.json`.

Run top to bottom, once, in one kernel. Cells 6 and 7 must not be re-run after cell 9:
the donor states and the donor-solved set are the shared spine of the whole comparison.

## VRAM budget (80 GB card)

| phase | resident | weights | activations | peak | headroom |
|---|---|---|---|---|---|
| D (donor only) | 27B bf16 | 54.5 GB | ~1.5 GB @ batch 8 | **~56 GB** | ~24 GB |
| A (recipient 9B) | 27B + 9B bf16 | 73.0 GB | ~2.5 GB @ batch 8 | **~75.5 GB** | **~4.5 GB** |
| free A | 27B bf16 | 54.5 GB | -- | ~55 GB | ~25 GB |
| B (recipient 2B) | 27B + 2B bf16 | 59.7 GB | ~2.5 GB @ batch 16 | **~62 GB** | ~18 GB |

Phase A is the tight one. Mitigations already applied: `ARITH_BATCH_A = 8`,
`expandable_segments:True` to limit fragmentation, hidden states moved to CPU per batch
(`.float().cpu()`, never accumulated on GPU), `torch.cuda.empty_cache()` at every phase
boundary, and `torch.cuda.memory_allocated()` printed at each boundary so headroom is
visible. If phase A still OOMs, set `ARITH_BATCH_A = 4` in CELL 2 and re-run from CELL 9
(the donor work in CELL 7 is unaffected -- `DONOR_BATCH` is separate and frozen).

## Reproducibility contract

`DONOR_BATCH = 8` for **every** donor forward, matching the two completed runs. This is
not cosmetic: left-padding depends on batch composition, and bf16 numerics depend on the
padding, so a different donor batch size silently changes which problems the donor solves.
`ARITH_BATCH` differs between the recipients (8 vs 16) and is never used for a donor
forward. The problem generator, the seeds (train 0, eval 1) and the sizes (3000 train /
2000 eval) are unchanged, so the donor-solved set should reproduce exactly. CELL 7 prints
`donor_solved_exprs_sha256` and warns loudly on a mismatch -- a mismatch means these
results are **not joinable** with the earlier runs.

## Scope

Arithmetic only (`a * b + c`). No GSM8K, no symbolic binding, no dose-response -- those
experiments already exist elsewhere in the suite.

In [1]:
# === CELL 1: Dependencies (run once, then RESTART THE KERNEL) ===
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer
!pip uninstall -y torchvision torchaudio
# torchvision/torchaudio are uninstalled BEFORE any torch import: on these pods their pinned
# versions disagree with the installed torch and the mismatch crashes `import transformers`.
# Nothing in this notebook uses them.
#
#   >>>>>>>>>>  RESTART THE KERNEL NOW, then continue at CELL 2.  <<<<<<<<<<
#
# --- Blackwell / sm_120 pods ONLY (CELL 2 prints the compute capability; if it says (12, 0)):
#     uncomment the line below, run it, and restart a second time. ---
!pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torchvision 0.23.0+cu128
Uninstalling torchvision-0.23.0+cu128:
  Successfully uninstalled torchvision-0.23.0+cu128
Found existing installation: torchaudio 2.8.0+cu128
Uninstalling torchaudio-2.8.0+cu128:
  Successfully uninstalled torchaudio-2.8.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached cuda_bindings-12.9.7-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 225.0 MB/s  0:00:03a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 211.2 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 219.9 MB/s  0:00:00a 0:00:01
  Using cached https://download-r2.pytorch.org/whl/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB

In [2]:
# === CELL 2: imports, set_submodule shim, global config ===
import os
# Set before torch initialises CUDA. Phase A leaves only ~4.5 GB of headroom on an 80 GB
# card, and allocator fragmentation is the likeliest way to lose it.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

import json, math, random, gc, hashlib, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)

# ---- the three models: ONE donor, TWO recipients evaluated sequentially ----
MODEL_DONOR   = "google/gemma-2-27b"   # 46 layers, d=4608, ~54.5 GB bf16
MODEL_RECIP_A = "google/gemma-2-9b"    # 42 layers, d=3584, ~18.5 GB bf16
MODEL_RECIP_B = "google/gemma-2-2b"    # 26 layers, d=2304, ~5.2 GB bf16

# ---- graft layers. Donor layer is FIXED across both recipients: the whole point is that
#      both recipients read the SAME donor state, so the donor layer must not vary. Each
#      recipient gets its own depth-matched layer (confirmed by CKA in CELL 9 / CELL 11).
L_DONOR   = 37   # start guess, shared by both phases
L_RECIP_A = 34   # start guess for gemma-2-9b
L_RECIP_B = 20   # start guess for gemma-2-2b

# candidate layers for the confirmation sweep (cheap: CKA on already-collected states)
L_DONOR_CANDIDATES   = [31, 34, 37, 40, 43]
L_RECIP_A_CANDIDATES = [28, 31, 34, 37]
L_RECIP_B_CANDIDATES = [14, 17, 20, 23]

# Confirmation only reports by default. Adopting a different recipient layer changes that
# recipient's native-solve set and therefore its unsolvable bin, which breaks the join with
# the two completed runs -- flip this only for a deliberate re-derivation.
ADOPT_CKA_BEST_RECIPIENT_LAYER = False

SMOKE_TEST = False   # True -> ~10 min end-to-end sanity pass, numbers are meaningless
if SMOKE_TEST:
    N_ARITH_TRAIN, N_ARITH_EVAL = 200, 120
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
    CKA_SUBSAMPLE = 64
else:
    N_ARITH_TRAIN, N_ARITH_EVAL = 3000, 2000    # identical to the two completed runs
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6
    BOOT_B = 10000
    CKA_SUBSAMPLE = 512

# ---- batch sizes ----
# DONOR_BATCH is FROZEN at 8 and used for every donor forward, exactly as in the two
# completed runs. Left-padding depends on batch composition and bf16 numerics depend on the
# padding, so changing this changes the donor-solved set. ARITH_BATCH is the RECIPIENT batch
# and legitimately differs between phases (VRAM); it is never used for a donor forward.
DONOR_BATCH   = 8
ARITH_BATCH_A = 8    # while the 9B recipient is resident (peak ~75.5 GB) -- drop to 4 on OOM
ARITH_BATCH_B = 16   # while the 2B recipient is resident (peak ~62 GB)
ARITH_BATCH   = ARITH_BATCH_A   # rebound at each phase boundary

MAX_NEW_ARITH = 8
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

# ---- reproducibility check against the two completed runs ----
# check below compares on the common prefix and says so explicitly rather than crying wolf.
EXPECTED_DONOR_SHA = "3986c0b753c7124a5d79e78b233989c57f9dadd18c49fb9825780ea12bd27523"

# ---- headline numbers from the two completed runs, for the side-by-side in CELL 12 ----
PRIOR_RUNS = {
    "9b": {"n": 240, "first": 0.725, "full": 0.120},
    "2b": {"n": 729, "first": 0.778, "full": 0.054},
}

OUT_JSON = "gemma27b_difficulty_matched_results.json"
RESULTS = {}

print("SMOKE_TEST =", SMOKE_TEST)
print(f"donor {MODEL_DONOR} L{L_DONOR} | recipients {MODEL_RECIP_A} L{L_RECIP_A} -> {MODEL_RECIP_B} L{L_RECIP_B}")
print(f"DONOR_BATCH={DONOR_BATCH} (frozen)  ARITH_BATCH_A={ARITH_BATCH_A}  ARITH_BATCH_B={ARITH_BATCH_B}")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0))
    print(f"total VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

SMOKE_TEST = False
donor google/gemma-2-27b L37 | recipients google/gemma-2-9b L34 -> google/gemma-2-2b L20
DONOR_BATCH=8 (frozen)  ARITH_BATCH_A=8  ARITH_BATCH_B=16
torch 2.11.0+cu128 | cuda True
device: NVIDIA RTX PRO 6000 Blackwell Server Edition | capability (12, 0)
total VRAM: 95.0 GB


In [ ]:
# === CELL 3: Hugging Face login (the Gemma weights are gated) ===
# Paste your own token below. Do NOT commit a real token, and rotate any token that has
# ever been pasted into a notebook you shared.
from huggingface_hub import login
login("")

In [4]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

def mcnemar_exact(a_bools, b_bools):
    """Exact (binomial) McNemar for PAIRED binary outcomes on the SAME items.

    This is the right test for the matched analysis: on the intersection bin the two
    recipients are scored on identical problems, so the samples are paired and two
    independent Wilson intervals would throw away that pairing. Returns
    (n_a0b1, n_a1b0, two_sided_p) over the discordant pairs only.
    """
    a = list(a_bools); b = list(b_bools)
    assert len(a) == len(b), "McNemar needs paired outcomes of equal length"
    n01 = sum(1 for x, y in zip(a, b) if (not x) and y)      # a wrong, b right
    n10 = sum(1 for x, y in zip(a, b) if x and (not y))      # a right, b wrong
    n = n01 + n10
    if n == 0: return n01, n10, 1.0
    k = min(n01, n10)
    p = sum(math.comb(n, i) for i in range(0, k+1)) / (2.0**n)
    return n01, n10, float(min(1.0, 2*p))

def vram(tag=""):
    """Print and return current GPU allocation in GB. Called at every phase boundary."""
    if not torch.cuda.is_available():
        print(f"[vram] {tag}: no cuda"); return 0.0
    a = torch.cuda.memory_allocated() / 2**30
    r = torch.cuda.memory_reserved() / 2**30
    t = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"[vram] {tag:<28s} allocated {a:6.2f} GB | reserved {r:6.2f} GB "
          f"| total {t:5.1f} GB | free~{t-r:5.2f} GB")
    return a

print("stats + vram helpers defined")

stats + vram helpers defined


In [5]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, full-answer) ===
# Copied from the verified 27B->2B notebook. Two deliberate changes, both noted inline:
#   (1) `batch` defaults to None instead of ARITH_BATCH, because a default argument binds at
#       def-time and ARITH_BATCH is rebound between phases -- a def-time default would freeze
#       the phase-A value and silently use it for phase B.
#   (2) the shuffle control's permutation is explicitly seeded (see CELL 8).
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(Xd, Xr, lam=RIDGE_LAMBDA):
    mud, mur = Xd.mean(0), Xr.mean(0)
    A, B = Xd-mud, Xr-mur
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mud, mur, W
def apply_map(x, m): mud, mur, W = m; return (x-mud)@W + mur

def linear_cka(X, Y):
    X, Y = X-X.mean(0, keepdim=True), Y-Y.mean(0, keepdim=True)
    hsic = (X.t()@Y).pow(2).sum()
    return (hsic / (torch.sqrt((X.t()@X).pow(2).sum())*torch.sqrt((Y.t()@Y).pow(2).sum()))).item()

# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
# UNCHANGED from the completed runs -- the donor-solved set depends on every character here.
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers the leading space fuses with the first digit, so it's at
    # len(p). Scanning for the first digit-bearing token handles both.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out

import re as _re
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None
def fd(x):
    """First decimal digit of an answer (used for the answer-shape sanity print)."""
    return int(str(abs(int(round(x))))[0]) if x is not None else 0

# ---- generic batched forward: last-pos resid at given layers + top token ----
# Hidden states are moved to CPU per batch (.float().cpu()); nothing accumulates on GPU.
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=None):
    batch = batch or ARITH_BATCH
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
        del out
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during PREFILL (seq len > 1); no-ops during generation (len == 1) and when the
# batch dim doesn't match, so generation proceeds normally after seeding. Every registration
# below is removed in a `finally`.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=None):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    batch = batch or ARITH_BATCH
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok

print("helpers defined")

helpers defined


In [6]:
# === CELL 5b: run plan, switches, and the per-problem export schema ===
# Everything that runs is gated here so the plan is visible before the 54 GB download starts.
RUN_PHASE_A = True    # recipient gemma-2-9b
RUN_PHASE_B = True    # recipient gemma-2-2b
RUN_LAYER_CONFIRM = True   # cheap CKA sweep; reports only unless ADOPT_... is True

# The per-problem export. One record per DONOR-SOLVED eval problem, keyed by the expression
# string, written for EACH recipient. Conferral is only defined where the donor solves the
# problem, so the donor-failed items carry no recipient outcome and are exported separately
# as a plain expression list (`donor_failed_exprs`) rather than as half-empty records.
PER_PROBLEM_SCHEMA = {
    "donor_solved":         "bool  -- True for every exported record (see note above)",
    "native_correct_first": "bool  -- recipient, no graft, top token == gold first answer token",
    "native_correct_full":  "bool  -- recipient, no graft, greedy generation matches the gold integer",
    "task_confer_first":    "bool  -- task-supervised map (seed 0), first token",
    "task_confer_full":     "bool  -- task-supervised map (seed 0), full answer",
    "recon_confer_first":   "bool  -- ridge reconstruction map, first token",
    "shuffle_confer_first": "bool  -- task map fed the WRONG problem's donor state (specificity control)",
    "gold_answer":          "int   -- a*b+c",
}
# Per-problem records use the seed-0 task map, because a per-problem outcome has to come from
# ONE map. The 5-seed across-seed aggregate on each recipient's own unsolvable bin is still
# computed and reported alongside, so the comparison with the completed runs stays honest.
PER_PROBLEM_TASK_SEED = 0
SHUFFLE_PERM_SEED = 1234   # the completed runs left this to global RNG state; seeding it
                           # makes the exported per-problem control reproducible.

RESULTS["config"] = {
    "model_donor": MODEL_DONOR, "model_recipient_a": MODEL_RECIP_A, "model_recipient_b": MODEL_RECIP_B,
    "L_donor": L_DONOR, "L_recipient_a": L_RECIP_A, "L_recipient_b": L_RECIP_B,
    "donor_batch": DONOR_BATCH, "arith_batch_a": ARITH_BATCH_A, "arith_batch_b": ARITH_BATCH_B,
    "n_arith_train": N_ARITH_TRAIN, "n_arith_eval": N_ARITH_EVAL,
    "task_seeds": TASK_SEEDS, "task_epochs": TASK_EPOCHS,
    "ridge_lambda": RIDGE_LAMBDA, "smoke_test": SMOKE_TEST,
    "per_problem_task_seed": PER_PROBLEM_TASK_SEED,
    "per_problem_schema": PER_PROBLEM_SCHEMA,
}
print(json.dumps({k: v for k, v in RESULTS["config"].items() if k != "per_problem_schema"}, indent=2))
print("\nper-problem export schema:")
for k, v in PER_PROBLEM_SCHEMA.items(): print(f"  {k:22s} {v}")

{
  "model_donor": "google/gemma-2-27b",
  "model_recipient_a": "google/gemma-2-9b",
  "model_recipient_b": "google/gemma-2-2b",
  "L_donor": 37,
  "L_recipient_a": 34,
  "L_recipient_b": 20,
  "donor_batch": 8,
  "arith_batch_a": 8,
  "arith_batch_b": 16,
  "n_arith_train": 3000,
  "n_arith_eval": 2000,
  "task_seeds": [
    0,
    1,
    2,
    3,
    4
  ],
  "task_epochs": 6,
  "ridge_lambda": 1000.0,
  "smoke_test": false,
  "per_problem_task_seed": 0
}

per-problem export schema:
  donor_solved           bool  -- True for every exported record (see note above)
  native_correct_first   bool  -- recipient, no graft, top token == gold first answer token
  native_correct_full    bool  -- recipient, no graft, greedy generation matches the gold integer
  task_confer_first      bool  -- task-supervised map (seed 0), first token
  task_confer_full       bool  -- task-supervised map (seed 0), full answer
  recon_confer_first     bool  -- ridge reconstruction map, first token
  shuffle_con

In [7]:
# === CELL 6: load the 27B DONOR once, in bf16 (NOT 4-bit) ===
# bf16, not 4-bit, deliberately: under 4-bit this transformers/bnb build runs the unquantised
# layers in fp16, Gemma-2 activations can exceed the fp16 range, and the result is NaN logits
# and an argmax that collapses to token 0. bf16 has the exponent range to avoid it. The health
# check below catches that failure immediately rather than 1800 silently-wrong problems later.
vram("before donor load")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DONOR)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model_d = AutoModelForCausalLM.from_pretrained(
    MODEL_DONOR, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
model_d.requires_grad_(False)
N_LAYERS_D = model_d.config.num_hidden_layers
print(f"donor loaded: {MODEL_DONOR} | {N_LAYERS_D} layers | d={model_d.config.hidden_size} | bf16")
vram("after donor load")

# Health check: catch a NaN/overflow blowup immediately. A healthy donor tops a real word.
with torch.inference_mode():
    _hl = model_d(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "donor produced NaN/Inf logits -- numerical blowup. The donor must be bf16, not 4-bit.")
print("donor health check OK | top token:", repr(tokenizer.decode([int(_hl.argmax())])))
del _hl

# Same-family requirement: all three models must share the TOKENIZER, because the graft is by
# position. config.vocab_size is the padded embedding count, not the tokenizer, so compare the
# tokenizers themselves on a probe string.
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
_ref = tokenizer(_probe).input_ids
for _name in (MODEL_RECIP_A, MODEL_RECIP_B):
    _tk = AutoTokenizer.from_pretrained(_name)
    assert _tk(_probe).input_ids == _ref, (
        f"tokenizer mismatch between {MODEL_DONOR} and {_name}: positions will not align. "
        "This notebook requires a same-family trio sharing one tokenizer.")
    del _tk
print("tokenizer identical across donor and both recipients")

# validate the configured layers against the real depths
assert 0 <= L_DONOR < N_LAYERS_D, f"L_DONOR={L_DONOR} out of range for {N_LAYERS_D} donor layers"
L_DONOR_CANDIDATES = [L for L in sorted(set(L_DONOR_CANDIDATES + [L_DONOR])) if 0 <= L < N_LAYERS_D]
print("donor candidate layers:", L_DONOR_CANDIDATES)

[vram] before donor load            allocated   0.00 GB | reserved   0.00 GB | total  95.0 GB | free~94.97 GB


model-00011-of-00024.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

model-00012-of-00024.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

model-00013-of-00024.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

model-00014-of-00024.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer identical across donor and both recipients
donor candidate layers: [31, 34, 37, 40, 43]


In [8]:
# === CELL 7: PHASE D -- the donor runs ONCE. Everything downstream reuses this. ===
# This is the expensive, shared, non-repeatable part: the problems, the donor states, and the
# donor-solved set. Both recipients are scored against exactly these, which is what makes the
# intersection bin in CELL 12 a valid difficulty-matched comparison.
vram("phase D start")

train = gen_arith(tokenizer, N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval (seeds 0 / 1, generator unchanged)")

# ---- donor forwards. DONOR_BATCH, always. Never ARITH_BATCH. ----
print(f"donor forward over train at DONOR_BATCH={DONOR_BATCH} ...")
Xd_train_all, _ = states_and_top(model_d, L_DONOR_CANDIDATES,
                                 [p["ids"] for p in train], batch=DONOR_BATCH)
print(f"donor forward over eval at DONOR_BATCH={DONOR_BATCH} ...")
Xd_eval_all, donor_top = states_and_top(model_d, L_DONOR_CANDIDATES,
                                        [p["ids"] for p in evalp], batch=DONOR_BATCH)

# ---- fix the donor-solved set ----
keep = [i for i in range(len(evalp)) if donor_top[i] == evalp[i]["tok"]]
donor_failed_exprs = [evalp[i]["expr"] for i in range(len(evalp)) if donor_top[i] != evalp[i]["tok"]]
donor_first_acc = len(keep) / len(evalp)
evalp = [evalp[i] for i in keep]
Xd_eval_all = {L: V[keep] for L, V in Xd_eval_all.items()}
Xd_train = Xd_train_all[L_DONOR]
Xd_eval  = Xd_eval_all[L_DONOR]
print(f"  donor first-token accuracy {donor_first_acc:.3f} -> kept {len(evalp)} donor-solved eval problems")
print(f"  donor states: train {tuple(Xd_train.shape)}, eval {tuple(Xd_eval.shape)} (CPU float32)")

# ---- reproducibility hash of the donor-solved set ----
donor_solved_exprs = [p["expr"] for p in evalp]
sha_kept   = hashlib.sha256("\n".join(donor_solved_exprs).encode()).hexdigest()
sha_sorted = hashlib.sha256("\n".join(sorted(donor_solved_exprs)).encode()).hexdigest()
_exp = EXPECTED_DONOR_SHA.strip().lower()
_match = any(s.startswith(_exp) or _exp.startswith(s) for s in (sha_kept, sha_sorted))

print("\n" + "="*78)
print("donor_solved_exprs_sha256 (generation order) =", sha_kept)
print("donor_solved_exprs_sha256 (sorted order)     =", sha_sorted)
print(f"expected (from the completed runs)           = {_exp}   [{len(_exp)} hex chars]")
if len(_exp) != 64:
    print(f"NOTE: the expected constant is {len(_exp)} characters, not the 64 of a full sha256, so it")
    print("      appears truncated. The check below compares on the common prefix.")
if _match:
    print("MATCH -- the donor-solved set reproduces. These results ARE joinable with the")
    print("        27B->9B and 27B->2B runs.")
else:
    print("!"*78)
    print("!!  LOUD WARNING: donor_solved_exprs_sha256 DOES NOT MATCH the completed runs.     !!")
    print("!!                                                                                !!")
    print("!!  The donor-solved set differs, so the per-problem records below are NOT         !!")
    print("!!  joinable with the earlier 27B->9B / 27B->2B results, and the prior-run column  !!")
    print("!!  in CELL 12 is NOT comparable to this run's numbers.                            !!")
    print("!!                                                                                !!")
    print("!!  The internal 9B-vs-2B comparison in this notebook is still valid: both         !!")
    print("!!  recipients share THIS donor set. Only the join to the old runs is broken.      !!")
    print("!!                                                                                !!")
    print("!!  Likely causes, in order: DONOR_BATCH changed from 8; a different transformers  !!")
    print("!!  or torch build; the donor loaded in 4-bit instead of bf16; a changed generator,!!")
    print("!!  seed (train 0 / eval 1) or size (3000 / 2000); a different donor checkpoint.   !!")
    print("!"*78)
print("="*78 + "\n")

_fdc = collections.Counter(fd(p["ans"]) for p in evalp)
print("first-digit distribution of the gold answers:",
      {d: _fdc[d] for d in sorted(_fdc)}, "(majority baseline "
      f"{max(_fdc.values())/len(evalp):.3f})")

RESULTS["donor"] = {
    "model": MODEL_DONOR, "layer": L_DONOR, "donor_batch": DONOR_BATCH,
    "n_eval_generated": N_ARITH_EVAL, "n_donor_solved": len(evalp),
    "donor_first_acc": round(donor_first_acc, 4),
    "donor_solved_exprs_sha256": sha_kept,
    "donor_solved_exprs_sha256_sorted": sha_sorted,
    "expected_sha_prefix": _exp,
    "sha_matches_completed_runs": bool(_match),
    "n_donor_failed": len(donor_failed_exprs),
}
RESULTS["donor_failed_exprs"] = donor_failed_exprs
vram("phase D done")

[vram] phase D start                allocated  50.72 GB | reserved  50.75 GB | total  95.0 GB | free~44.22 GB
arith: 3000 train, 2000 eval (seeds 0 / 1, generator unchanged)
donor forward over train at DONOR_BATCH=8 ...
donor forward over eval at DONOR_BATCH=8 ...
  donor first-token accuracy 0.894 -> kept 1789 donor-solved eval problems
  donor states: train (3000, 4608), eval (1789, 4608) (CPU float32)

donor_solved_exprs_sha256 (generation order) = 0567bf97c3ae99fe147721cd5783b24f74dee63340a7eecccff50e2ffba4d01b
donor_solved_exprs_sha256 (sorted order)     = fadf054b2b3fbab886c1e8e371ba121568364dfcf49c1646dd9e3e321e7d4339
expected (from the completed runs)           = 3986c0b753c7124a5d79e78b233989c57f9dadd18c49fb9825780ea12bd27523   [64 hex chars]
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!  LOUD WARNING: donor_solved_exprs_sha256 DOES NOT MATCH the completed runs.     !!
!!                                                                       

50.722434997558594

In [9]:
# === CELL 8: the recipient engine (definitions only -- nothing runs here) ===
# One implementation, called twice. Both recipients therefore go through byte-identical code;
# the only differences are the model, the graft layer and the batch size. Every GPU tensor a
# phase creates is local to these functions and released before the phase returns, which is
# what makes the "free recipient A completely" check in CELL 10 pass.

def confirm_layers(tag, model_r, recip_cands, donor_cands, Xd_all, probs, batch, n=None):
    """Cheap CKA confirmation of the graft pair. Reports; does not change anything unless
    ADOPT_CKA_BEST_RECIPIENT_LAYER is set. CKA is a representational-similarity heuristic,
    not a causal test -- treat a disagreement with the configured layer as a prompt to look,
    not as an instruction to move the graft."""
    n = n or CKA_SUBSAMPLE
    n = min(n, len(probs))
    nl = model_r.config.num_hidden_layers
    cands = [L for L in sorted(set(recip_cands)) if 0 <= L < nl]
    print(f"[{tag}] CKA layer confirmation on {n} train problems, recipient layers {cands} ...")
    st, _ = states_and_top(model_r, cands, [p["ids"] for p in probs[:n]], batch=batch)
    table = {}
    for lr in cands:
        table[lr] = {ld: round(linear_cka(st[lr], Xd_all[ld][:n]), 3) for ld in donor_cands}
    hdr = "  recip \\ donor  " + "".join(f"  L{ld:<5d}" for ld in donor_cands)
    print(hdr); print("  " + "-"*(len(hdr)-2))
    for lr in cands:
        print(f"  L{lr:<13d}" + "".join(f"  {table[lr][ld]:<6.3f}" for ld in donor_cands))
    best = max(((lr, ld, table[lr][ld]) for lr in cands for ld in donor_cands), key=lambda t: t[2])
    best_for_fixed_donor = max(((lr, table[lr][L_DONOR]) for lr in cands), key=lambda t: t[1])
    print(f"  best overall pair: recip L{best[0]} <-> donor L{best[1]}  CKA={best[2]:.3f}")
    print(f"  best recipient layer at the FIXED donor L{L_DONOR}: L{best_for_fixed_donor[0]} "
          f"CKA={best_for_fixed_donor[1]:.3f}")
    del st
    gc.collect(); torch.cuda.empty_cache()
    return {"table": {str(k): v for k, v in table.items()},
            "best_pair": [best[0], best[1], best[2]],
            "best_recipient_layer_at_fixed_donor": [best_for_fixed_donor[0], best_for_fixed_donor[1]]}


def train_task_maps(tag, model_r, L_r, Xd_tr, mu_d_dev, W_init, b_init, train_probs, batch):
    """The task-supervised map: the only stochastic part. Recipient params are frozen, so the
    autograd graph starts at the grafted vector and covers only the layers above L_r."""
    maps = []
    model_r.requires_grad_(False)
    for seed in TASK_SEEDS:
        torch.manual_seed(seed); random.seed(seed)
        W = W_init.clone().to(DEVICE).requires_grad_(True)
        b = b_init.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=1e-3)
        handle = model_r.model.layers[L_r].register_forward_hook(patch_vec_batch)
        idx = list(range(len(train_probs)))
        loss = None
        try:
            for ep in range(TASK_EPOCHS):
                random.Random(seed*100+ep).shuffle(idx)
                for s in range(0, len(idx), batch):
                    sub = idx[s:s+batch]
                    xd = Xd_tr[sub].to(DEVICE)
                    _graft["vec"] = (xd - mu_d_dev) @ W + b
                    ids, m = left_pad([train_probs[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([train_probs[k]["tok"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
        finally:
            handle.remove(); _graft["vec"] = None
        maps.append((W.detach(), b.detach()))
        print(f"    [{tag}] seed {seed}: final batch CE {loss.item():.3f}")
    model_r.requires_grad_(False)
    return maps


@torch.inference_mode()
def first_token_confer(model_r, L_r, map_fn, Xd_ev, probs, idxs, batch):
    handle = model_r.model.layers[L_r].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), batch):
            sub = idxs[i:i+batch]
            _graft["vec"] = map_fn(Xd_ev[sub])
            ids, m = left_pad([probs[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == probs[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out


@torch.inference_mode()
def shuffle_confer(model_r, L_r, task_maps, mu_d_dev, Xd_ev, probs, idxs, shuf, batch):
    """Specificity control: the task map is fed the WRONG problem's donor state."""
    handle = model_r.model.layers[L_r].register_forward_hook(patch_vec_batch)
    out = []
    W, b = task_maps[0]
    try:
        for i in range(0, len(idxs), batch):
            sub = idxs[i:i+batch]
            donor = Xd_ev[shuf[i:i+len(sub)]].to(DEVICE)
            _graft["vec"] = (donor - mu_d_dev)@W + b
            ids, m = left_pad([probs[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == probs[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out


def full_confer(model_r, L_r, map_fn, Xd_ev, probs, idxs, batch):
    with torch.inference_mode():
        vecs = list(map_fn(Xd_ev[idxs]).detach().cpu())
    return arith_fullanswer_correct(model_r, L_r, [probs[j] for j in idxs], vecs=vecs, batch=batch)


def evaluate_recipient(tag, model_r, L_r, batch, Xd_tr, Xd_ev, train_probs, eval_probs):
    """Full evaluation of one recipient against the shared donor states.

    Returns (per_problem_records, summary). Every outcome in the schema is computed over the
    WHOLE donor-solved eval set, not just that recipient's own unsolvable bin -- that is
    precisely what the completed runs failed to save and what the matched analysis needs.
    """
    t_all = list(range(len(eval_probs)))
    print(f"\n[{tag}] recipient states @ L{L_r}, batch={batch} ...")
    st_t, _ = states_and_top(model_r, [L_r], [p["ids"] for p in train_probs], batch=batch)
    Xr_tr = st_t[L_r]; del st_t
    st_e, r_top = states_and_top(model_r, [L_r], [p["ids"] for p in eval_probs], batch=batch)
    Xr_ev = st_e[L_r]; del st_e
    gc.collect(); torch.cuda.empty_cache()

    native_first = [r_top[i] == eval_probs[i]["tok"] for i in t_all]
    unsolv = [i for i in t_all if not native_first[i]]
    solv   = [i for i in t_all if native_first[i]]
    print(f"[{tag}] natively solves {len(solv)}/{len(eval_probs)} at the first token "
          f"(unsolvable bin n={len(unsolv)}, {len(unsolv)/max(1,len(eval_probs)):.1%} of the donor-solved set)")

    # ---- reconstruction map (ridge, donor -> recipient state) ----
    mu_d, mu_r, W_r = fit_ridge(Xd_tr, Xr_tr)
    mu_d_dev, mu_r_dev, W_r_dev = mu_d.to(DEVICE), mu_r.to(DEVICE), W_r.to(DEVICE)
    def map_recon(xd): return (xd.to(DEVICE) - mu_d_dev) @ W_r_dev + mu_r_dev

    # ---- task-supervised map, 5 seeds ----
    print(f"[{tag}] training the task map, {len(TASK_SEEDS)} seeds x {TASK_EPOCHS} epochs ...")
    task_maps = train_task_maps(tag, model_r, L_r, Xd_tr, mu_d_dev, W_r, mu_r, train_probs, batch)
    def map_task(i):
        W, b = task_maps[i]
        return lambda xd: (xd.to(DEVICE) - mu_d_dev) @ W + b
    vram(f"{tag} after map training")

    shuf = torch.randperm(len(eval_probs),
                          generator=torch.Generator().manual_seed(SHUFFLE_PERM_SEED))

    # ---- the per-problem outcomes, over the WHOLE donor-solved set ----
    print(f"[{tag}] native full-answer (no graft) over all {len(t_all)} ...")
    native_full = arith_fullanswer_correct(model_r, L_r, eval_probs, vecs=None, batch=batch)
    print(f"[{tag}] task map (seed {PER_PROBLEM_TASK_SEED}) first token ...")
    task_first = first_token_confer(model_r, L_r, map_task(PER_PROBLEM_TASK_SEED), Xd_ev, eval_probs, t_all, batch)
    print(f"[{tag}] task map (seed {PER_PROBLEM_TASK_SEED}) full answer ...")
    task_full = full_confer(model_r, L_r, map_task(PER_PROBLEM_TASK_SEED), Xd_ev, eval_probs, t_all, batch)
    print(f"[{tag}] reconstruction map first token ...")
    recon_first = first_token_confer(model_r, L_r, map_recon, Xd_ev, eval_probs, t_all, batch)
    print(f"[{tag}] shuffle control first token ...")
    shuf_first = shuffle_confer(model_r, L_r, task_maps, mu_d_dev, Xd_ev, eval_probs, t_all, shuf, batch)

    records = {}
    for i, p in enumerate(eval_probs):
        records[p["expr"]] = {
            "donor_solved": True,
            "native_correct_first": bool(native_first[i]),
            "native_correct_full": bool(native_full[i]),
            "task_confer_first": bool(task_first[i]),
            "task_confer_full": bool(task_full[i]),
            "recon_confer_first": bool(recon_first[i]),
            "shuffle_confer_first": bool(shuf_first[i]),
            "gold_answer": int(p["ans"]),
        }
    assert len(records) == len(eval_probs), "expression collision in the per-problem export"

    # ---- aggregates on this recipient's OWN unsolvable bin (comparable to the old runs) ----
    def sel(v, ii): return [v[i] for i in ii]
    summary = {"model": tag, "layer_recipient": L_r, "layer_donor": L_DONOR,
               "arith_batch": batch, "n_eval_donor_solved": len(eval_probs),
               "n_unsolvable_own_bin": len(unsolv), "n_solvable": len(solv),
               "recon_cos_recon": fmt(bootstrap_ci(
                   F.cosine_similarity(map_recon(Xd_ev).cpu(), Xr_ev, dim=1).numpy())),
               "recon_cos_task": fmt(bootstrap_ci(
                   F.cosine_similarity(map_task(0)(Xd_ev).detach().cpu(), Xr_ev, dim=1).numpy()))}
    if unsolv:
        summary["native_unsolv_first"] = fmt(wilson_bools(sel(native_first, unsolv)))
        summary["native_unsolv_full"] = fmt(wilson_bools(sel(native_full, unsolv)))
        summary["recon_unsolv_first"] = fmt(wilson_bools(sel(recon_first, unsolv)))
        summary["task_unsolv_first_pooled"] = fmt(wilson_bools(sel(task_first, unsolv)))
        summary["task_unsolv_full_seed0"] = fmt(wilson_bools(sel(task_full, unsolv)))
        summary["shuffle_unsolv_first"] = fmt(wilson_bools(sel(shuf_first, unsolv)))
        # the 5-seed across-seed interval, exactly as the completed runs reported it
        print(f"[{tag}] 5-seed full-answer on the own unsolvable bin (n={len(unsolv)}) ...")
        seed_full = [float(np.mean(full_confer(model_r, L_r, map_task(s), Xd_ev, eval_probs, unsolv, batch)))
                     for s in range(len(task_maps))]
        summary["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
        summary["task_unsolv_full_per_seed"] = [round(x, 4) for x in seed_full]
        summary["recon_unsolv_full"] = fmt(wilson_bools(
            full_confer(model_r, L_r, map_recon, Xd_ev, eval_probs, unsolv, batch)))
    if solv:
        summary["native_solv_full"] = fmt(wilson_bools(sel(native_full, solv)))
        summary["task_solv_first"] = fmt(wilson_bools(sel(task_first, solv)))
        summary["task_solv_full"] = fmt(wilson_bools(sel(task_full, solv)))

    # ---- release every GPU tensor this phase created ----
    del task_maps, W_r_dev, mu_d_dev, mu_r_dev, map_recon, map_task, Xr_tr, Xr_ev
    _graft["vec"] = None
    gc.collect(); torch.cuda.empty_cache()
    print(f"[{tag}] done.")
    return records, summary

print("recipient engine defined")

recipient engine defined


In [10]:
# === CELL 9: PHASE A -- recipient gemma-2-9b (peak ~75.5 GB with the donor resident) ===
per_problem_9b, summary_9b = None, None
if RUN_PHASE_A:
    ARITH_BATCH = ARITH_BATCH_A
    vram("phase A before recipient load")
    model_r_a = AutoModelForCausalLM.from_pretrained(
        MODEL_RECIP_A, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
    model_r_a.requires_grad_(False)
    print(f"recipient A loaded: {MODEL_RECIP_A} | {model_r_a.config.num_hidden_layers} layers "
          f"| d={model_r_a.config.hidden_size}")
    vram("phase A donor + 9B resident")
    assert 0 <= L_RECIP_A < model_r_a.config.num_hidden_layers

    if RUN_LAYER_CONFIRM:
        cka_a = confirm_layers("9b", model_r_a, L_RECIP_A_CANDIDATES, L_DONOR_CANDIDATES,
                               Xd_train_all, train, ARITH_BATCH)
        RESULTS["layer_confirm_9b"] = cka_a
        _best_a = cka_a["best_recipient_layer_at_fixed_donor"][0]
        if _best_a != L_RECIP_A:
            print(f"  NOTE: CKA prefers recipient L{_best_a} over the configured L{L_RECIP_A}.")
            if ADOPT_CKA_BEST_RECIPIENT_LAYER:
                L_RECIP_A = _best_a; print(f"  ADOPTED L_RECIP_A = {L_RECIP_A}")
            else:
                print("  keeping the configured layer (ADOPT_CKA_BEST_RECIPIENT_LAYER is False)")

    per_problem_9b, summary_9b = evaluate_recipient(
        "9b", model_r_a, L_RECIP_A, ARITH_BATCH, Xd_train, Xd_eval, train, evalp)
    RESULTS["recipient_9b"] = summary_9b
    print("\n[9b] summary:", json.dumps(summary_9b, indent=2))
    vram("phase A done (recipient still resident)")
else:
    print("RUN_PHASE_A is False -- skipping recipient A")

[vram] phase A before recipient load allocated  50.72 GB | reserved  51.76 GB | total  95.0 GB | free~43.21 GB


config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/4.84G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

model-00007-of-00008.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/2.38G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

recipient A loaded: google/gemma-2-9b | 42 layers | d=3584
[vram] phase A donor + 9B resident  allocated  67.94 GB | reserved  67.96 GB | total  95.0 GB | free~27.02 GB
[9b] CKA layer confirmation on 512 train problems, recipient layers [28, 31, 34, 37] ...
  recip \ donor    L31     L34     L37     L40     L43   
  -------------------------------------------------------
  L28             0.705   0.700   0.601   0.640   0.569 
  L31             0.747   0.728   0.631   0.646   0.580 
  L34             0.839   0.850   0.840   0.804   0.761 
  L37             0.737   0.786   0.880   0.860   0.869 
  best overall pair: recip L37 <-> donor L37  CKA=0.880
  best recipient layer at the FIXED donor L37: L37 CKA=0.880
  NOTE: CKA prefers recipient L37 over the configured L34.
  keeping the configured layer (ADOPT_CKA_BEST_RECIPIENT_LAYER is False)

[9b] recipient states @ L34, batch=8 ...
[9b] natively solves 1550/1789 at the first token (unsolvable bin n=239, 13.4% of the donor-solved set)
[9b

In [11]:
# === CELL 10: free recipient A COMPLETELY before recipient B loads ===
# The 80 GB budget only works because the two recipients are never resident at once. This
# cell asserts that, rather than trusting it.
_before = vram("before freeing recipient A")
for _n in ["model_r_a"]:
    if _n in globals():
        del globals()[_n]
_graft["vec"] = None
# Jupyter keeps the last expression value in _, __, ___ and Out[]; those can pin a model.
for _n in ["_", "__", "___"]:
    if _n in globals(): globals()[_n] = None
try:
    get_ipython().user_ns["Out"].clear()
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache(); gc.collect(); torch.cuda.empty_cache()
_after = vram("after freeing recipient A")
_dropped = _before - _after
print(f"allocation dropped by {_dropped:.2f} GB (gemma-2-9b in bf16 is ~18.5 GB)")
if RUN_PHASE_A:
    assert _dropped > 10.0, (
        f"recipient A was NOT fully freed: allocation only dropped {_dropped:.2f} GB. Something "
        "still holds a reference to the model. Do not load recipient B -- you will OOM. Check "
        "for stray references (a cell output, a closure, a debugger frame) and re-run this cell.")
    print("recipient A fully freed; only the donor remains resident.")

[vram] before freeing recipient A   allocated  68.13 GB | reserved  68.21 GB | total  95.0 GB | free~26.77 GB
[vram] after freeing recipient A    allocated  50.73 GB | reserved  50.76 GB | total  95.0 GB | free~44.21 GB
allocation dropped by 17.40 GB (gemma-2-9b in bf16 is ~18.5 GB)
recipient A fully freed; only the donor remains resident.


In [16]:
print(type(per_problem_9b), len(per_problem_9b) if per_problem_9b else "NONE")

<class 'dict'> 1789


In [17]:
!df -h / /workspace
!du -sh /root/.cache/huggingface/hub/* 2>/dev/null | sort -h | tail

Filesystem                   Size  Used Avail Use% Mounted on
overlay                      190G  4.4G  186G   3% /
mfs#us-nc-1.runpod.net:9421  1.2P  878T  324T  74% /workspace


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
!rm -rf /root/.cache/huggingface/hub/models--google--gemma-2-9b
!find /root/.cache/huggingface -name "*.incomplete" -delete
!pip cache purge
!df -h / /workspace

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


find: ‘/root/.cache/huggingface’: No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Files removed: 194 (1171.4 MB)
Filesystem                   Size  Used Avail Use% Mounted on
overlay                      190G  4.4G  186G   3% /
mfs#us-nc-1.runpod.net:9421  1.2P  878T  324T  74% /workspace


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
import json
json.dump({"per_problem_9b": per_problem_9b, "summary_9b": summary_9b,
           "donor": RESULTS.get("donor", {})}, open("/workspace/phaseA_backup.json", "w"))
print("bytes:", __import__("os").path.getsize("/workspace/phaseA_backup.json"))

OSError: [Errno 122] Disk quota exceeded

In [20]:
import json, os
json.dump({"per_problem_9b": per_problem_9b, "summary_9b": summary_9b,
           "donor": RESULTS.get("donor", {})}, open("/root/phaseA_backup.json", "w"))
print("bytes:", os.path.getsize("/root/phaseA_backup.json"))

bytes: 416777


In [21]:
!env | grep -i -E "hf_home|hf_hub|transformers_cache"
!du -sh /workspace/* 2>/dev/null | sort -h | tail

HF_HUB_ENABLE_HF_TRANSFER=1
HF_HOME=/workspace/.cache/huggingface/


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


0	/workspace/phaseA_backup.json
137K	/workspace/80GB_27B_DifficultyMatched.ipynb


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
!rm -rf /workspace/<cache_path>/hub/models--google--gemma-2-9b

/bin/bash: line 1: cache_path: No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [23]:
import os
os.environ["HF_HOME"] = "/root/hf"
os.environ["HF_HUB_CACHE"] = "/root/hf/hub"

In [24]:
!du -sh /workspace/.cache/huggingface/hub/* 2>/dev/null | sort -h | tail
!rm -rf /workspace/.cache/huggingface/hub/models--google--gemma-2-9b
!du -sh /workspace/.cache/huggingface 2>/dev/null

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


512	/workspace/.cache/huggingface/hub/version.txt
2.8G	/workspace/.cache/huggingface/hub/models--google--gemma-2-2b
35G	/workspace/.cache/huggingface/hub/models--google--gemma-2-9b
102G	/workspace/.cache/huggingface/hub/models--google--gemma-2-27b


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


105G	/workspace/.cache/huggingface


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [25]:
!rm -rf /workspace/.cache/huggingface/hub/models--google--gemma-2-2b
!find /workspace/.cache/huggingface -name "*.incomplete" -delete

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
# === CELL 11: PHASE B -- recipient gemma-2-2b (peak ~62 GB with the donor resident) ===
per_problem_2b, summary_2b = None, None
if RUN_PHASE_B:
    ARITH_BATCH = ARITH_BATCH_B
    vram("phase B before recipient load")
    model_r_b = AutoModelForCausalLM.from_pretrained(
        MODEL_RECIP_B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
    model_r_b.requires_grad_(False)
    print(f"recipient B loaded: {MODEL_RECIP_B} | {model_r_b.config.num_hidden_layers} layers "
          f"| d={model_r_b.config.hidden_size}")
    vram("phase B donor + 2B resident")
    assert 0 <= L_RECIP_B < model_r_b.config.num_hidden_layers

    if RUN_LAYER_CONFIRM:
        cka_b = confirm_layers("2b", model_r_b, L_RECIP_B_CANDIDATES, L_DONOR_CANDIDATES,
                               Xd_train_all, train, ARITH_BATCH)
        RESULTS["layer_confirm_2b"] = cka_b
        _best_b = cka_b["best_recipient_layer_at_fixed_donor"][0]
        if _best_b != L_RECIP_B:
            print(f"  NOTE: CKA prefers recipient L{_best_b} over the configured L{L_RECIP_B}.")
            if ADOPT_CKA_BEST_RECIPIENT_LAYER:
                L_RECIP_B = _best_b; print(f"  ADOPTED L_RECIP_B = {L_RECIP_B}")
            else:
                print("  keeping the configured layer (ADOPT_CKA_BEST_RECIPIENT_LAYER is False)")

    per_problem_2b, summary_2b = evaluate_recipient(
        "2b", model_r_b, L_RECIP_B, ARITH_BATCH, Xd_train, Xd_eval, train, evalp)
    RESULTS["recipient_2b"] = summary_2b
    print("\n[2b] summary:", json.dumps(summary_2b, indent=2))
    vram("phase B done")
else:
    print("RUN_PHASE_B is False -- skipping recipient B")

[vram] phase B before recipient load allocated  50.73 GB | reserved  50.76 GB | total  95.0 GB | free~44.21 GB


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

recipient B loaded: google/gemma-2-2b | 26 layers | d=2304
[vram] phase B donor + 2B resident  allocated  55.60 GB | reserved  55.61 GB | total  95.0 GB | free~39.36 GB
[2b] CKA layer confirmation on 512 train problems, recipient layers [14, 17, 20, 23] ...
  recip \ donor    L31     L34     L37     L40     L43   
  -------------------------------------------------------
  L14             0.531   0.531   0.450   0.499   0.434 
  L17             0.618   0.613   0.531   0.559   0.494 
  L20             0.653   0.665   0.646   0.624   0.570 
  L23             0.630   0.684   0.744   0.709   0.701 
  best overall pair: recip L23 <-> donor L37  CKA=0.744
  best recipient layer at the FIXED donor L37: L23 CKA=0.744
  NOTE: CKA prefers recipient L23 over the configured L20.
  keeping the configured layer (ADOPT_CKA_BEST_RECIPIENT_LAYER is False)

[2b] recipient states @ L20, batch=16 ...
[2b] natively solves 1069/1789 at the first token (unsolvable bin n=720, 40.2% of the donor-solved set)
[2

In [27]:
# === CELL 12: THE DIFFICULTY-MATCHED ANALYSIS ===
# Both recipients were scored on the same donor-solved problems with the same donor states, so
# the per-problem records can be joined on the expression string. The intersection bin holds
# difficulty fixed: every problem in it is one that BOTH recipients fail natively.
assert per_problem_9b is not None and per_problem_2b is not None, (
    "both phases must have run: re-run CELL 9 and CELL 11 with RUN_PHASE_A / RUN_PHASE_B True")
k9, k2 = set(per_problem_9b), set(per_problem_2b)
assert k9 == k2, (f"the two recipients were scored on different problem sets "
                  f"({len(k9)} vs {len(k2)}, {len(k9 ^ k2)} differ) -- the join is invalid. "
                  "Both phases must reuse the CELL 7 donor-solved set; do not re-run CELL 7.")
ALL_EXPRS = sorted(k9)

unsolv_9b = {e for e in ALL_EXPRS if not per_problem_9b[e]["native_correct_first"]}
unsolv_2b = {e for e in ALL_EXPRS if not per_problem_2b[e]["native_correct_first"]}
intersection_bin = sorted(unsolv_9b & unsolv_2b)
union_bin = sorted(unsolv_9b | unsolv_2b)

def rate(recs, exprs, field):
    v = [recs[e][field] for e in exprs]
    return wilson_bools(v) if v else (float("nan"),)*3
def vals(recs, exprs, field):
    return [recs[e][field] for e in exprs]

FIELDS = [("task_confer_first",   "first-token conferral"),
          ("task_confer_full",    "full-answer conferral"),
          ("recon_confer_first",  "recon first-token"),
          ("shuffle_confer_first","shuffle first-token"),
          ("native_correct_full", "native full-answer")]

print("="*96)
print("DIFFICULTY-MATCHED COMPARISON: 27B donor -> {9B, 2B} recipients")
print("="*96)
print(f"donor-solved eval problems (shared by both recipients): {len(ALL_EXPRS)}")
print(f"  9B fails natively at the first token: {len(unsolv_9b):5d}  ({len(unsolv_9b)/len(ALL_EXPRS):.1%})")
print(f"  2B fails natively at the first token: {len(unsolv_2b):5d}  ({len(unsolv_2b)/len(ALL_EXPRS):.1%})")
print(f"  INTERSECTION (both fail)            : {len(intersection_bin):5d}  ({len(intersection_bin)/len(ALL_EXPRS):.1%})")
print(f"  union (either fails)                : {len(union_bin):5d}")
_nested = unsolv_9b <= unsolv_2b
print(f"  is the 9B bin a subset of the 2B bin? {_nested}  "
      f"(9B-only items: {len(unsolv_9b - unsolv_2b)})")
print()
print("This is the confound in one line: the two 'unsolvable bins' above are different problem")
print("sets, and the 9B's is the harder one. Everything below is on the SAME problems.")

print("\n" + "="*96)
print(f"A. MATCHED -- intersection bin, n = {len(intersection_bin)} (identical problems for both)")
print("="*96)
print(f"  {'metric':<26s} {'9B':<26s} {'2B':<26s} {'McNemar p (paired)':<18s}")
print("  " + "-"*92)
matched = {"n": len(intersection_bin)}
for f, label in FIELDS:
    r9, r2 = rate(per_problem_9b, intersection_bin, f), rate(per_problem_2b, intersection_bin, f)
    n01, n10, p = mcnemar_exact(vals(per_problem_9b, intersection_bin, f),
                                vals(per_problem_2b, intersection_bin, f))
    print(f"  {label:<26s} {fmt(r9):<26s} {fmt(r2):<26s} {p:<18.2e}")
    matched[f] = {"9b": fmt(r9), "2b": fmt(r2), "mcnemar_p": p,
                  "discordant_9b_only": n10, "discordant_2b_only": n01,
                  "diff_2b_minus_9b": round(r2[0]-r9[0], 4)}

print("\n" + "="*96)
print("B. UNMATCHED -- each recipient on its OWN unsolvable bin (how the confound was created)")
print("="*96)
print(f"  {'metric':<26s} {'9B (own bin)':<26s} {'2B (own bin)':<26s}")
print("  " + "-"*80)
unmatched = {"n_9b": len(unsolv_9b), "n_2b": len(unsolv_2b)}
for f, label in FIELDS:
    r9 = rate(per_problem_9b, sorted(unsolv_9b), f)
    r2 = rate(per_problem_2b, sorted(unsolv_2b), f)
    print(f"  {label:<26s} {fmt(r9):<26s} {fmt(r2):<26s}")
    unmatched[f] = {"9b": fmt(r9), "2b": fmt(r2), "diff_2b_minus_9b": round(r2[0]-r9[0], 4)}

print("\n" + "="*96)
print("C. SIDE BY SIDE -- prior runs vs this run's own bins vs this run matched")
print("="*96)
print(f"  {'':<22s} {'prior run':<14s} {'this run, own bin':<20s} {'this run, MATCHED':<20s}")
print("  " + "-"*80)
_mf9, _mf2 = matched["task_confer_first"], matched["task_confer_full"]
rows = [
    ("9B  n",              PRIOR_RUNS['9b']['n'],     len(unsolv_9b),                    len(intersection_bin)),
    ("2B  n",              PRIOR_RUNS['2b']['n'],     len(unsolv_2b),                    len(intersection_bin)),
    ("9B  first-token",    PRIOR_RUNS['9b']['first'], rate(per_problem_9b, sorted(unsolv_9b), 'task_confer_first')[0], rate(per_problem_9b, intersection_bin, 'task_confer_first')[0]),
    ("2B  first-token",    PRIOR_RUNS['2b']['first'], rate(per_problem_2b, sorted(unsolv_2b), 'task_confer_first')[0], rate(per_problem_2b, intersection_bin, 'task_confer_first')[0]),
    ("9B  full-answer",    PRIOR_RUNS['9b']['full'],  rate(per_problem_9b, sorted(unsolv_9b), 'task_confer_full')[0],  rate(per_problem_9b, intersection_bin, 'task_confer_full')[0]),
    ("2B  full-answer",    PRIOR_RUNS['2b']['full'],  rate(per_problem_2b, sorted(unsolv_2b), 'task_confer_full')[0],  rate(per_problem_2b, intersection_bin, 'task_confer_full')[0]),
]
for label, a, b, c in rows:
    if isinstance(a, int): print(f"  {label:<22s} {a:<14d} {b:<20d} {c:<20d}")
    else:                  print(f"  {label:<22s} {a:<14.3f} {b:<20.3f} {c:<20.3f}")
if not RESULTS["donor"]["sha_matches_completed_runs"]:
    print("\n  CAUTION: the donor-solved SHA did not match, so the 'prior run' column is not")
    print("           strictly comparable to the other two. See the CELL 7 warning.")
print("\n  Note on the full-answer column: the per-problem records use the seed-0 task map,")
print("  because a per-problem outcome needs a single map. The completed runs reported a")
print("  5-seed across-seed mean. Both are in the saved JSON; the matched comparison is")
print("  seed-0 vs seed-0, which is the like-for-like contrast.")
print(f"  this run, 5-seed across-seed on own bins: "
      f"9B {summary_9b.get('task_unsolv_full_acrossseed')} | 2B {summary_2b.get('task_unsolv_full_acrossseed')}")

# ------------------------------ the verdict ------------------------------
print("\n" + "="*96)
print("D. VERDICT -- does each dissociation survive difficulty matching?")
print("="*96)
f9 = rate(per_problem_9b, intersection_bin, "task_confer_first")[0]
f2 = rate(per_problem_2b, intersection_bin, "task_confer_first")[0]
g9 = rate(per_problem_9b, intersection_bin, "task_confer_full")[0]
g2 = rate(per_problem_2b, intersection_bin, "task_confer_full")[0]
p_first = matched["task_confer_first"]["mcnemar_p"]
p_full  = matched["task_confer_full"]["mcnemar_p"]
ALPHA = 0.05

first_dir = f2 >= f9
first_sig = p_first < ALPHA
full_dir  = g9 > g2
full_sig  = p_full < ALPHA

def _verdict(direction, sig):
    if direction and sig: return "SURVIVES (direction holds, paired test significant)"
    if direction and not sig: return "DIRECTION HOLDS BUT IS NOT SIGNIFICANT (underpowered / no real effect)"
    return "DOES NOT SURVIVE (direction reverses or vanishes under matching)"

print(f"  FIRST-TOKEN dissociation (prior claim: 2B >= 9B, {PRIOR_RUNS['2b']['first']:.3f} vs {PRIOR_RUNS['9b']['first']:.3f})")
print(f"    matched: 2B {f2:.3f} vs 9B {f9:.3f}  (2B - 9B = {f2-f9:+.3f}), McNemar p = {p_first:.2e}")
print(f"    VERDICT: {_verdict(first_dir, first_sig)}")
print()
print(f"  FULL-ANSWER dissociation (prior claim: 9B > 2B, {PRIOR_RUNS['9b']['full']:.3f} vs {PRIOR_RUNS['2b']['full']:.3f})")
print(f"    matched: 9B {g9:.3f} vs 2B {g2:.3f}  (9B - 2B = {g9-g2:+.3f}), McNemar p = {p_full:.2e}")
print(f"    VERDICT: {_verdict(full_dir, full_sig)}")
print()
if first_dir and full_dir:
    print("  BOTH halves of the dissociation survive matching: the pattern is a recipient effect,")
    print("  not an artefact of the 9B being scored on a harder subset. The reading that the first")
    print("  token is transferred content while later digits are the recipient's own computation")
    print("  stands, now on identical problems.")
elif full_dir and not first_dir:
    print("  Only the FULL-ANSWER half survives. The apparent first-token advantage of the weaker")
    print("  recipient was the difficulty confound: on matched problems it reverses or flattens.")
elif first_dir and not full_dir:
    print("  Only the FIRST-TOKEN half survives. The full-answer gap was the difficulty confound.")
else:
    print("  NEITHER half survives. The published dissociation was an artefact of comparing the two")
    print("  recipients on different, unequally hard problem sets.")
print("\n  Read the verdict with the shuffle and recon rows above: if shuffle conferral is not")
print("  near the floor on the intersection bin, the graft is not carrying problem-specific")
print("  content there and the conferral numbers do not mean what the labels say.")

RESULTS["difficulty_matched"] = {
    "n_donor_solved": len(ALL_EXPRS),
    "n_unsolv_9b": len(unsolv_9b), "n_unsolv_2b": len(unsolv_2b),
    "n_intersection": len(intersection_bin), "n_union": len(union_bin),
    "nine_bin_subset_of_two_bin": bool(_nested),
    "matched": matched, "unmatched_own_bins": unmatched,
    "prior_runs_reported": PRIOR_RUNS,
    "verdict": {
        "first_token_2b_ge_9b": bool(first_dir), "first_token_mcnemar_p": p_first,
        "first_token_survives": bool(first_dir and first_sig),
        "full_answer_9b_gt_2b": bool(full_dir), "full_answer_mcnemar_p": p_full,
        "full_answer_survives": bool(full_dir and full_sig),
        "alpha": ALPHA,
    },
    "intersection_bin_exprs": intersection_bin,
}

DIFFICULTY-MATCHED COMPARISON: 27B donor -> {9B, 2B} recipients
donor-solved eval problems (shared by both recipients): 1789
  9B fails natively at the first token:   239  (13.4%)
  2B fails natively at the first token:   720  (40.2%)
  INTERSECTION (both fail)            :   153  (8.6%)
  union (either fails)                :   806
  is the 9B bin a subset of the 2B bin? False  (9B-only items: 86)

This is the confound in one line: the two 'unsolvable bins' above are different problem
sets, and the 9B's is the harder one. Everything below is on the SAME problems.

A. MATCHED -- intersection bin, n = 153 (identical problems for both)
  metric                     9B                         2B                         McNemar p (paired)
  --------------------------------------------------------------------------------------------
  first-token conferral      0.719 [0.643, 0.784]       0.693 [0.616, 0.760]       4.24e-01          
  full-answer conferral      0.118 [0.076, 0.178]       0.0

In [28]:
# === CELL 13: save everything, including the per-problem exports ===
RESULTS["per_problem_9b"] = per_problem_9b or {}
RESULTS["per_problem_2b"] = per_problem_2b or {}
with open(OUT_JSON, "w") as f:
    json.dump(RESULTS, f, indent=2)

import os as _os
print(f"saved {OUT_JSON}  ({_os.path.getsize(OUT_JSON)/1e6:.2f} MB)")
print(f"  per_problem_9b: {len(RESULTS['per_problem_9b'])} records")
print(f"  per_problem_2b: {len(RESULTS['per_problem_2b'])} records")
print("  top-level keys:", sorted(RESULTS.keys()))
_ex = next(iter(RESULTS["per_problem_9b"].items()), None)
if _ex:
    print("\nexample record:")
    print(f"  {_ex[0]!r} -> {json.dumps(_ex[1])}")
print("\nTo join these with a future run: key on the expression string, and check that")
print("RESULTS['donor']['donor_solved_exprs_sha256'] matches.")

saved gemma27b_difficulty_matched_results.json  (1.06 MB)
  per_problem_9b: 1789 records
  per_problem_2b: 1789 records
  top-level keys: ['config', 'difficulty_matched', 'donor', 'donor_failed_exprs', 'layer_confirm_2b', 'layer_confirm_9b', 'per_problem_2b', 'per_problem_9b', 'recipient_2b', 'recipient_9b']

example record:
  '6 * 18 + 16' -> {"donor_solved": true, "native_correct_first": true, "native_correct_full": false, "task_confer_first": true, "task_confer_full": false, "recon_confer_first": true, "shuffle_confer_first": true, "gold_answer": 124}

To join these with a future run: key on the expression string, and check that
RESULTS['donor']['donor_solved_exprs_sha256'] matches.
